In [ ]:
# --- Xray Cloud: Release Test Metrics with Parquet Caching ---

import requests, getpass, os, datetime, pandas as pd, urllib3
from datetime import datetime as dt

# ============================================================
# Global Config
# ============================================================

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
CACHE_DIR = "./xray_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

XRAY_AUTH_URL    = "https://xray.cloud.getxray.app/api/v2/authenticate"
XRAY_GRAPHQL_URL = "https://xray.cloud.getxray.app/api/v2/graphql"
VERIFY_SSL = False  

# ============================================================
# Auth
# ============================================================

CID  = getpass.getpass("XRAY_CLIENT_ID: ")
CSEC = getpass.getpass("XRAY_CLIENT_SECRET: ")

def auth():
    """Authenticate and return token."""
    payload = {"client_id": CID, "client_secret": CSEC}
    print("Authenticating to Xray Cloud...")
    r = requests.post(XRAY_AUTH_URL, json=payload, timeout=30, verify=VERIFY_SSL)
    if r.status_code != 200:
        print("Authentication failed:", r.text)
        raise SystemExit("Check Client ID/Secret or environment.")
    return r.text.strip('"')

TOKEN   = auth()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

# ============================================================
# Helper: GraphQL Wrapper
# ============================================================

def gql(query, variables=None):
    """Execute GraphQL query."""
    r = requests.post(
        XRAY_GRAPHQL_URL,
        json={"query": query, "variables": variables or {}},
        headers=HEADERS,
        timeout=60,
        verify=VERIFY_SSL,
    )
    r.raise_for_status()
    data = r.json()
    if "errors" in data:
        raise SystemExit(f"GraphQL errors: {data['errors']}")
    return data["data"]

# ============================================================
# Status Normalization
# ============================================================

STATUS_MAP = {
    "PASSED": "PASSED", "PASS": "PASSED", "OK": "PASSED",
    "FAILED": "FAILED", "FAIL": "FAILED",
    "TO DO": "TO DO", "TODO": "TO DO", "NOT_RUN": "TO DO", "UNTESTED": "TO DO",
    "BLOCKED": "BLOCKED", "EXECUTING": "EXECUTING",
    "DESCOPED": "DESCOPED", "SKIPPED": "DESCOPED", "WONT DO": "DESCOPED"
}

def norm_status(s: str) -> str:
    return STATUS_MAP.get((s or "").strip().upper(), s)

# ============================================================
# Helper: Save to Cache
# ============================================================

def cache_path_for(plan_key): 
    return os.path.join(CACHE_DIR, f"{plan_key}.parquet")

def save_to_cache(plan_key, df):
    path = cache_path_for(plan_key)
    df.to_parquet(path, index=False)
    print(f"Saved data for {plan_key} → {path}")

# ============================================================
# Helper: Get Test Plan Status
# ============================================================

def get_test_plan_status(plan_key):
    """Return the Jira/Xray status of the given Test Plan."""
    Q_STATUS = """
    query($jql:String!, $limit:Int!, $start:Int!) {
      getTestPlans(jql:$jql, limit:$limit, start:$start) {
        results {
          jira(fields:["key","summary","status"])
        }
      }
    }
    """
    d = gql(Q_STATUS, {"jql": f'key = "{plan_key}"', "limit": 1, "start": 0})
    plans = d.get("getTestPlans", {}).get("results", [])
    if not plans:
        return "Unknown"

    status_field = plans[0]["jira"].get("status")
    if isinstance(status_field, dict):
        return status_field.get("name", "Unknown").strip().title()
    elif isinstance(status_field, str):
        return status_field.strip().title()
    else:
        return "Unknown"


# ============================================================
# Fetch Planned Test Keys
# ============================================================

def get_planned_test_keys_from_plan(plan_id: str):
    """Fetch unique test keys directly from Xray Test Plan."""
    Q_PLAN_TESTS_FROM_PLAN = """
    query($planId:String!, $limit:Int!, $start:Int!) {
      getTestPlan(issueId:$planId) {
        tests(limit:$limit, start:$start) {
          total start limit
          results { jira(fields:["key"]) }
        }
      }
    }
    """
    planned = set()
    start = 0
    page = 200
    while True:
        d = gql(Q_PLAN_TESTS_FROM_PLAN, {"planId": plan_id, "limit": page, "start": start})
        block = d["getTestPlan"]["tests"]
        for t in block["results"]:
            key = (t.get("jira") or {}).get("key")
            if key:
                planned.add(key)
        start += block["limit"]
        if start >= block["total"]:
            break
    return planned

# ============================================================
# Fetch Release Data (Planned + Execution)
# ============================================================

def fetch_release_data(plan_key):
    print(f"Fetching data for {plan_key} ...")

    # --- 1) Resolve Plan Info ---
    Q_PLANS = """
    query($jql:String!, $limit:Int!, $start:Int!) {
      getTestPlans(jql:$jql, limit:$limit, start:$start) {
        results { issueId jira(fields:["key","summary","status"]) }
      }
    }
    """
    plans = gql(Q_PLANS, {"jql": f'key = "{plan_key}"', "limit": 1, "start": 0})
    plan_info = plans["getTestPlans"]["results"][0]
    plan_id = plan_info["issueId"]
    plan_name = plan_info["jira"]["summary"]
    plan_status = plan_info["jira"]["status"]

    # --- 2) Planned Test Scope ---
    planned_keys = get_planned_test_keys_from_plan(plan_id)
    planned_scope = [
        {"plan_key": plan_key, "plan_name": plan_name, "test_key": key, "source": "PLANNED_SCOPE", "status": "TO DO"}
        for key in planned_keys
    ]

    # --- 3) Execution Runs ---
    Q_PLAN_EXECS = """
    query($planId:String!, $limit:Int!, $start:Int!) {
      getTestPlan(issueId:$planId) {
        testExecutions(limit:$limit, start:$start) {
          total start limit
          results { issueId }
        }
      }
    }
    """
    exec_ids, start, page = [], 0, 100
    while True:
        d = gql(Q_PLAN_EXECS, {"planId": plan_id, "limit": page, "start": start})
        te = d["getTestPlan"]["testExecutions"]
        exec_ids += [x["issueId"] for x in te["results"]]
        start += te["limit"]
        if start >= te["total"]:
            break

    Q_RUNS = """
    query($execIds:[String], $limit:Int!, $start:Int!) {
      getTestRuns(testExecIssueIds:$execIds, limit:$limit, start:$start) {
        total start limit
        results {
          status { name }
          test { jira(fields:["key"]) }
        }
      }
    }
    """
    all_runs = []
    if exec_ids:
        BATCH = 50
        for i in range(0, len(exec_ids), BATCH):
            batch = exec_ids[i:i+BATCH]
            start = 0
            limit = 100
            while True:
                data = gql(Q_RUNS, {"execIds": batch, "limit": limit, "start": start})
                runs = data["getTestRuns"]
                for tr in runs["results"]:
                    test_key = (tr.get("test") or {}).get("jira", {}).get("key")
                    sname = (tr.get("status") or {}).get("name") or "UNKNOWN"
                    all_runs.append({
                        "plan_key": plan_key,
                        "plan_name": plan_name,
                        "plan_status": plan_status,
                        "test_key": test_key,
                        "source": "EXECUTION_RUN",
                        "status": norm_status(sname)
                    })
                start += runs["limit"]
                if start >= runs["total"]:
                    break

    combined = pd.DataFrame(planned_scope + all_runs)
    combined["timestamp"] = datetime.datetime.now().isoformat()
    return combined

# ============================================================
# Dynamic Cache Logic
# ============================================================

def fetch_or_load(plan_key):
    """Smart cache loader that refreshes if plan is not Done/Closed."""
    path = cache_path_for(plan_key)
    plan_status = get_test_plan_status(plan_key)
    print(f"Plan {plan_key} status: {plan_status}")

    # Normalize status (case-insensitive)
    plan_status_norm = (plan_status or "").strip().lower()

    # Define what counts as "finalized"
    done_statuses = {"done", "closed", "completed"}
    
    # If not finalized → always fetch fresh data
    if plan_status_norm not in done_statuses:
        print(f"Plan is active ({plan_status}) → fetching fresh data from Xray...")
        df = fetch_release_data(plan_key)
        save_to_cache(plan_key, df)
        return df

    # If finalized & cache exists → reuse cached data
    if os.path.exists(path):
        try:
            cached = pd.read_parquet(path)
            print(f"Using cached data for {plan_key} (status = {plan_status})")
            return cached
        except Exception as e:
            print(f"Cache load failed for {plan_key}: {e} → refetching...")
            df = fetch_release_data(plan_key)
            save_to_cache(plan_key, df)
            return df

    # If finalized but no cache yet → fetch once and cache
    print(f" No cache found for {plan_key} (status = {plan_status}) → fetching fresh data...")
    df = fetch_release_data(plan_key)
    save_to_cache(plan_key, df)
    return df


# ============================================================
# Compute Release Metrics (All 6 Statuses)
# ============================================================
def compute_release_metrics(df_all):
    """Compute release metrics across all six standard statuses, with status normalization."""
    summary = []

    for rel in df_all["plan_key"].unique():
        subset = df_all[df_all["plan_key"] == rel]

        # --- Normalize plan_status safely ---
        plan_status = "Unknown"
        if "plan_status" in subset.columns:
            # Convert dicts or other weird values into strings
            normalized = subset["plan_status"].apply(
                lambda x: x.get("name") if isinstance(x, dict)
                else str(x).strip().title() if isinstance(x, str)
                else None
            ).dropna()
            unique_statuses = normalized.unique()
            if len(unique_statuses) > 0:
                plan_status = unique_statuses[0]

        # --- Planned scope ---
        planned = subset[subset["source"] == "PLANNED_SCOPE"]["test_key"].nunique()

        # --- Execution runs ---
        exec_subset = subset[subset["source"] == "EXECUTION_RUN"]
        actual_execution = len(exec_subset)

        # --- Unique outcomes ---
        unique_status = exec_subset.groupby("test_key")["status"].last().value_counts()
        status_list = ["TO DO", "PASSED", "FAILED", "BLOCKED", "EXECUTING", "DESCOPED"]
        counts = {s: unique_status.get(s, 0) for s in status_list}

        todo, passed, failed, blocked, executing, descoped = (
            counts["TO DO"], counts["PASSED"], counts["FAILED"],
            counts["BLOCKED"], counts["EXECUTING"], counts["DESCOPED"]
        )

        # --- Duplicated run analysis ---
        run_counts = exec_subset["test_key"].value_counts()
        duplicated_runs = (run_counts > 1).sum()

        # --- Metrics ---
        executed_tests = planned - todo
        pass_rate = round((passed / (passed + failed) * 100), 2) if (passed + failed) else 0
        exec_coverage = round((executed_tests / planned * 100), 2) if planned else 0
        completion = round(((passed + failed + descoped + blocked + executing) / planned * 100), 2) if planned else 0
        remaining = round(((planned - (passed + failed + descoped + blocked + executing)) / planned * 100), 2) if planned else 0

        summary.append({
            "Release": rel,
            "Status": plan_status,
            "Planned Tests": planned,
            "Actual Execution": actual_execution,
            "Duplicated Runs": duplicated_runs,
            "Execution Coverage %": exec_coverage,
            "Pass %": pass_rate,
            "Completion %": completion,
            "Remaining %": remaining,
            "To Do": todo,
            "Passed": passed,
            "Failed": failed,
            "Blocked": blocked,
            "Executing": executing,
            "Descoped": descoped
        })

    cols = [
        "Release", "Status", "Planned Tests", "Actual Execution", "Duplicated Runs",
        "Execution Coverage %", "Pass %", "Completion %", "Remaining %",
        "To Do", "Passed", "Failed", "Blocked", "Executing", "Descoped"
    ]
    return pd.DataFrame(summary)[cols]
# ============================================================
# Run for Releases
# ============================================================

release_keys = [
    "PROJ-1003","PROJ-1001","PROJ-1002", "PROJ-1004" , "PROJ-1005"
]

all_dfs = [fetch_or_load(rk) for rk in release_keys]
df_all = pd.concat(all_dfs, ignore_index=True)

dashboard_df = compute_release_metrics(df_all)
print("\n=== Release Test Metrics Dashboard ===\n")
print(dashboard_df.to_string(index=False))

dashboard_df.to_csv("xray_cache/release_metrics_dashboard.csv", index=False)
dashboard_df.to_parquet("xray_cache/release_metrics_dashboard.parquet", index=False)
print("\n Dashboard successfully generated.")
